# Speech-to-Text: 음성 파일을 텍스트로 전사하기

STT(Speech-to-Text) 또는 ASR(Automatic Speech Recognition)은 음성의 말을 텍스트로 바꾸는 기능이다. 회의록 초안, 자막 생성, 음성 검색에 사용할 수 있다.

이 노트북은 MP3 파일을 준비하고 전사 API에 전달한다. 02_tts.ipynb에서 만든 `output.mp3`를 사용하면 음성 생성과 전사가 하나의 흐름으로 연결된다.

현재 파일 전사의 주요 선택지는 다음과 같다.

- 일반 파일 전사는 `gpt-transcribe`로 시작한다.
- 입력 파일은 25MB 이하여야 한다.
- 지원 형식은 `mp3`, `mp4`, `mpeg`, `mpga`, `m4a`, `wav`, `webm`이다.
- 실시간 마이크 입력은 Realtime transcription을 사용한다.
- 화자 구분은 `gpt-4o-transcribe-diarize`를 사용한다.

### 요청 비용과 재실행

- `gpt-transcribe`: 전사하는 음성 1분당 0.0045달러이다. 단순 비례 계산으로 10분은 약 0.045달러이다.
- 같은 1분 파일을 기본 전사와 문맥 힌트 전사로 두 번 요청하면 단순 합계는 약 0.009달러이다.
- 같은 파일을 다시 실행할 때마다 새 API 요청이 발생하므로 짧은 예제 음성을 사용한다.
- 가격은 변경될 수 있으므로 [GPT Transcribe 모델 가격](https://developers.openai.com/api/docs/models/gpt-transcribe)과 [API 가격표](https://developers.openai.com/api/docs/pricing#transcription-models)를 실행 전에 확인한다.

공식 문서는 다음과 같다.

- [Speech-to-Text 가이드](https://developers.openai.com/api/docs/guides/speech-to-text)
- [Transcription 생성 API](https://developers.openai.com/api/reference/resources/audio/subresources/transcriptions/methods/create)
- [모델 카탈로그](https://developers.openai.com/api/docs/models)


## Whisper와 파일 전사 모델 선택

Whisper는 다국어 음성 인식과 음성 번역에 널리 쓰인 기반 모델 계열이다. 최신 API에서는 목적에 따라 모델을 나누어 선택한다.

- 일반적인 완성 파일 전사는 `gpt-transcribe`를 우선 사용한다.
- 단어·구간 타임스탬프가 필요하면 `whisper-1`을 사용한다.
- 영어 번역 엔드포인트도 `whisper-1`을 사용한다.
- 화자 이름과 구간이 필요하면 `gpt-4o-transcribe-diarize`를 사용한다.
- 계속 들어오는 음성은 파일 전사 대신 Realtime transcription을 사용한다.


### API 클라이언트 준비

이미 설정한 `.env`를 불러온다. 키 값은 출력하지 않으며, 성공하면 다음 전사 셀의 `OpenAI()`가 환경 변수에서 인증 정보를 읽는다.

- `find_dotenv(usecwd=True)`: 현재 작업 폴더부터 상위 폴더에서 `.env` 경로를 찾는다.
- `load_dotenv(dotenv_path, override=False)`: 찾은 값을 커널 환경 변수에 등록하되, 이미 설정된 환경 변수는 덮어쓰지 않는다.
- `os.getenv('OPENAI_API_KEY')`: 키 값 자체를 출력하지 않고 다음 API 요청에 필요한 변수의 존재만 확인한다.


In [1]:
import os

from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위의 .env 파일을 확인한다.")
load_dotenv(dotenv_path, override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env의 OPENAI_API_KEY를 확인한다.")

print("환경 변수 준비 완료")


환경 변수 준비 완료


### 전사할 음성 파일 선택하기

먼저 TTS 실습에서 만든 `output.mp3`를 찾는다. 파일이 없을 때만 예제 음성을 `stt_sample.mp3`로 내려받는다.

- `output.mp3`: TTS 생성 결과를 전사한다.
- `stt_sample.mp3`: 독립적으로 STT만 진행할 때 사용한다.
- 두 파일명을 분리해 TTS 결과를 덮어쓰지 않는다.

- `Path('output.mp3')`: TTS가 만든 상대 경로를 `file_path`로 만든다. `exists()`가 참이면 이 경로를 그대로 다음 전사 셀에 전달한다.
- `gdown --output stt_sample.mp3 <file-id>`: TTS 파일이 없을 때만 Google Drive 파일 ID `1zHD6wDwqGyDzAp9Y254SZEruRALiK9Rw`의 예제 음성을 별도 이름으로 받는다.
- `subprocess.run(..., check=True)`: 다운로드 명령이 실패하면 예외를 발생시켜 존재하지 않는 파일을 전사하지 않게 한다.


In [2]:
from pathlib import Path
import subprocess

file_path = Path("output.mp3")

### 음성 파일을 `gpt-transcribe`로 전사하기

`open(file_path, 'rb')`는 MP3 바이트를 읽기 모드로 열고, 전사 API는 이 파일을 텍스트 응답으로 변환한다. `transcription.text`가 최종 전사 문자열이며 자막, 요약, 검색 같은 다음 텍스트 처리의 입력으로 사용할 수 있다.

- `client.audio.transcriptions.create(model='gpt-transcribe', file=f)`: `file` 바이트를 기본 전사 모델에 보내 `transcription` 응답을 받는다.
- `transcription.text`: 최종 전사 문자열이며 화면 출력 뒤 요약·검색·자막 생성의 입력으로 사용한다.
- 다음 실습에서는 `prompt`, `keywords`, `languages`를 추가해 같은 음성의 기본 전사 결과와 비교한다.
- `stream=True`는 완료된 파일을 처리하는 동안 부분 전사 이벤트가 필요할 때만 추가한다.

파일 크기와 지원 형식은 [Speech-to-Text 공식 가이드](https://developers.openai.com/api/docs/guides/speech-to-text)에서 수업 당일 다시 확인한다.


In [3]:
from openai import OpenAI

client = OpenAI()

with open(file_path, 'rb') as f:
    transcription = client.audio.transcriptions.create(
        model='gpt-transcribe',
        file=f
    )

print(transcription.text)

다들 오늘 점심 어떤 거 드시나요? 추천 좀 해주세요.


In [6]:
print(transcription)

Transcription(text='다들 오늘 점심 어떤 거 드시나요? 추천 좀 해주세요.', languages=[TranscriptionLanguage(code='ko')], logprobs=None, usage=UsageDuration(seconds=4.0, type='duration'))


In [7]:
def create_text(file_path):
    with open(file_path, 'rb') as f:
        transcription = client.audio.transcriptions.create(
            model='gpt-transcribe',
            file=f
        )
        return transcription.text


In [11]:
# print( create_text("slow.mp3") )
# print( create_text("normal.mp3") )
# print( create_text("fast.mp3") )
print( create_text("high-fast.mp3") )

Like me, like me. 아주 눈이 보시는 너를 숨김없이 보여줘. 한 번도 빛난 적 없었던 BGA 향으로 온 세상을 물들여. 새로워진 장면에 두 눈 앞은 황홀해. 너의 손을 잡을 땐 너와 어우러질 때 빛을 이끌어. With me, with me. 마치 chemically, 우리 완벽하게 어울려. Feeling love attack.


## 문맥 힌트로 전사 정확도 높이기

음성이 짧거나 발음이 불분명하면 모델이 고유명사와 전문 용어를 일반 단어로 바꿔 적을 수 있다. `gpt-transcribe`는 녹음의 배경, 예상 용어, 가능한 언어를 힌트로 받아 후보 범위를 좁힐 수 있다. 힌트는 정답을 강제하지 않으므로 실제 발화와 결과를 반드시 비교해야 한다.

- `prompt`: 녹음의 주제와 상황을 자유로운 문장으로 전달한다. 음성과 같은 언어로 작성한다.
- `keywords`: 실제로 들릴 가능성이 높은 고유명사나 전문 용어를 문자열 목록으로 전달한다. 발화하지 않은 단어를 과도하게 넣으면 잘못 끼워 넣을 수 있다.
- `languages`: 음성에 포함될 수 있는 언어를 ISO 639-1 코드 목록으로 전달한다. 한국어는 `ko`, 영어는 `en`이며 번역할 출력 언어가 아니라 입력 음성의 후보 언어이다.
- Python SDK에서는 `prompt`를 직접 전달하고, `keywords`와 `languages`는 `extra_body`에 넣는다. 이 두 필드는 `gpt-transcribe`에서 지원한다.

아래 예시는 앞에서 사용한 음성이 날씨와 건강에 관한 한국어 안내라고 가정한다. 다른 음성 파일을 사용한다면 세 힌트를 실제 녹음 내용에 맞게 수정한다. 자세한 필드 정의는 [File transcription 가이드](https://developers.openai.com/api/docs/guides/speech-to-text#add-transcription-context)와 [Transcription 생성 API](https://developers.openai.com/api/reference/resources/audio/subresources/transcriptions/methods/create)에서 확인할 수 있다.


In [14]:

# 전사하는 음성파일의 주제와 상황을 자연어 문장으로 설명한다.
transcription_prompt = "K-POP 여자 아이돌 노래 가사이고, 한글과 영어가 혼합되어있다. 가수 이름은 '리센느', 노래 제목은 'Love Attack'이다."

# keywords: 실제 발화에서 정확히 표기하고 싶은 단어나 구를 작성
expected_keywords = ["미지", '부시는']

# 입력 음성에서 예상되는 언어 코드 목록
expected_languages = ["ko"]

with open("high-fast.mp3", "rb") as f:
    guided_transcription = client.audio.transcriptions.create(
        model="gpt-transcribe",
        file=f,
        prompt=transcription_prompt,
        extra_body={
            "keywords": expected_keywords,
            "languages": expected_languages
        }
    )

# 응답의 languages가 비어 있을 수 있으므로 빈 목록을 기본값으로 사용해 언어 코드만 추출한다.
detected_language_codes = [
    language.code for language in (guided_transcription.languages or [])
]

# 출력: 힌트 적용 전후 문자열과 모델이 감지한 언어 코드이다.
print("기본 전사:", transcription.text)
print("문맥 힌트 적용:", guided_transcription.text)
print("감지 언어:", detected_language_codes)

기본 전사: 다들 오늘 점심 어떤 거 드시나요? 추천 좀 해주세요.
문맥 힌트 적용: Like me, like me. 아주 눈이 부신 너를 숨김없이 보여줘. 한 번도 빛난 적 없었던 미지의 향으로 온 세상을 물들여. 새로워진 장면에 두 눈 앞은 황홀해. 너의 손을 잡을 땐 너와 어우러질 때 빛을 이끌어. With me, with me. 마치 chemically, 우린 완벽하게 어울려. Feeling love attack.
감지 언어: ['ko']
